# A9: APR Timeline Tracking

**Purpose**: Create permit timeline tracking tables for HCD Annual Progress Report generation.

## HCD APR Requirements
California Housing and Community Development (HCD) requires annual reporting on:
- Date application submitted
- Date deemed complete
- Date entitlement issued
- Date building permit issued
- Date certificate of occupancy
- Units by income category (Very Low, Low, Moderate, Above Moderate)

## Data Sources
1. **Berkeley Accela Portal** - https://aca-prod.accela.com/BERKELEY/
2. **Building Eye** - https://berkeley.buildingeye.com/
3. **Our existing project data** - 157 projects with addresses

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime
import requests
from bs4 import BeautifulSoup
import time
import re

# Paths
ROOT = Path.cwd().parent if Path.cwd().name == '01_collection' else Path.cwd()
DB_PATH = ROOT / 'databases' / 'berkeley_housing_map.db'
ADDRESS_DB = ROOT / 'databases' / 'berkeley_address_centric.db'

print(f"Database: {DB_PATH}")
print(f"Exists: {DB_PATH.exists()}")

## 1. Create Timeline Tracking Tables

In [ ]:
# Connect to database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create permit_applications table
cursor.execute('''
CREATE TABLE IF NOT EXISTS permit_applications (
    permit_id INTEGER PRIMARY KEY,
    project_id INTEGER REFERENCES projects(project_id),
    address_id INTEGER,
    
    -- Identifiers (APR fields 1-4)
    apn TEXT,
    street_address TEXT,
    project_name TEXT,
    local_tracking_id TEXT,
    
    -- Permit Type
    permit_type TEXT,
    permit_subtype TEXT,
    
    -- Timeline Dates (APR fields 5-9)
    date_submitted DATE,
    date_deemed_complete DATE,
    date_entitlement_issued DATE,
    date_building_permit_issued DATE,
    date_certificate_occupancy DATE,
    
    -- Current Status
    current_status TEXT,
    
    -- Metadata
    data_source TEXT,
    source_url TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

# Create indexes
cursor.execute('CREATE INDEX IF NOT EXISTS idx_permit_project ON permit_applications(project_id)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_permit_status ON permit_applications(current_status)')

print("Created permit_applications table")

In [ ]:
# Create permit_timeline_events table
cursor.execute('''
CREATE TABLE IF NOT EXISTS permit_timeline_events (
    event_id INTEGER PRIMARY KEY,
    permit_id INTEGER REFERENCES permit_applications(permit_id),
    
    event_type TEXT NOT NULL,
    event_date DATE NOT NULL,
    event_description TEXT,
    
    days_since_previous INTEGER,
    reviewer_department TEXT,
    
    data_source TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

cursor.execute('CREATE INDEX IF NOT EXISTS idx_event_permit ON permit_timeline_events(permit_id)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_event_date ON permit_timeline_events(event_date)')

print("Created permit_timeline_events table")

In [ ]:
# Create affordability_tracking table
cursor.execute('''
CREATE TABLE IF NOT EXISTS affordability_tracking (
    affordability_id INTEGER PRIMARY KEY,
    project_id INTEGER REFERENCES projects(project_id),
    permit_id INTEGER REFERENCES permit_applications(permit_id),
    
    -- Units by Income Category (APR Fields 10-14)
    units_acutely_low INTEGER DEFAULT 0,
    units_extremely_low INTEGER DEFAULT 0,
    units_very_low INTEGER DEFAULT 0,
    units_low INTEGER DEFAULT 0,
    units_moderate INTEGER DEFAULT 0,
    units_above_moderate INTEGER DEFAULT 0,
    
    -- Tenure (APR Field 15)
    tenure TEXT,
    
    -- Affordability Mechanism (APR Fields 16-18)
    deed_restricted BOOLEAN DEFAULT FALSE,
    affordability_term_years INTEGER,
    funding_source TEXT,
    
    verified_date DATE,
    verification_source TEXT,
    
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

cursor.execute('CREATE INDEX IF NOT EXISTS idx_afford_project ON affordability_tracking(project_id)')

print("Created affordability_tracking table")

In [ ]:
# Create apr_submissions tracking table
cursor.execute('''
CREATE TABLE IF NOT EXISTS apr_submissions (
    submission_id INTEGER PRIMARY KEY,
    reporting_year INTEGER NOT NULL,
    
    submission_date DATE,
    submitted_to TEXT DEFAULT 'HCD',
    
    total_projects INTEGER,
    total_units INTEGER,
    units_entitled INTEGER,
    units_permitted INTEGER,
    units_completed INTEGER,
    
    -- RHNA Progress
    rhna_very_low_target INTEGER,
    rhna_very_low_progress INTEGER,
    rhna_low_target INTEGER,
    rhna_low_progress INTEGER,
    rhna_moderate_target INTEGER,
    rhna_moderate_progress INTEGER,
    rhna_above_moderate_target INTEGER,
    rhna_above_moderate_progress INTEGER,
    
    apr_file_path TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

conn.commit()
print("Created apr_submissions table")

In [ ]:
# Create APR export view
cursor.execute('DROP VIEW IF EXISTS apr_table_a2_view')
cursor.execute('''
CREATE VIEW apr_table_a2_view AS
SELECT
    pa.apn AS "APN",
    pa.street_address AS "Street Address",
    pa.project_name AS "Project Name",
    pa.local_tracking_id AS "Local Jurisdiction Tracking ID",
    strftime('%m/%d/%Y', pa.date_submitted) AS "Date Application Submitted",
    strftime('%m/%d/%Y', pa.date_deemed_complete) AS "Date Deemed Complete",
    strftime('%m/%d/%Y', pa.date_entitlement_issued) AS "Date Entitlement Issued",
    strftime('%m/%d/%Y', pa.date_building_permit_issued) AS "Date Building Permit Issued",
    strftime('%m/%d/%Y', pa.date_certificate_occupancy) AS "Date Certificate of Occupancy",
    COALESCE(af.units_very_low, 0) AS "Very Low Income Units",
    COALESCE(af.units_low, 0) AS "Low Income Units",
    COALESCE(af.units_moderate, 0) AS "Moderate Income Units",
    COALESCE(af.units_above_moderate, 0) AS "Above Moderate Income Units",
    af.tenure AS "Tenure",
    CASE WHEN af.deed_restricted THEN 'Yes' ELSE 'No' END AS "Deed Restricted",
    af.affordability_term_years AS "Term of Affordability",
    af.funding_source AS "Funding Source"
FROM permit_applications pa
LEFT JOIN affordability_tracking af ON pa.permit_id = af.permit_id
WHERE pa.date_building_permit_issued IS NOT NULL
   OR pa.date_certificate_occupancy IS NOT NULL
   OR pa.date_entitlement_issued IS NOT NULL
ORDER BY pa.date_building_permit_issued DESC
''')

conn.commit()
print("Created apr_table_a2_view")

In [ ]:
# Create processing times view
cursor.execute('DROP VIEW IF EXISTS permit_processing_times')
cursor.execute('''
CREATE VIEW permit_processing_times AS
SELECT
    pa.permit_id,
    pa.street_address,
    pa.permit_type,
    pa.date_submitted,
    pa.date_deemed_complete,
    pa.date_building_permit_issued,
    pa.date_certificate_occupancy,
    julianday(pa.date_deemed_complete) - julianday(pa.date_submitted) AS days_to_complete,
    julianday(pa.date_building_permit_issued) - julianday(pa.date_deemed_complete) AS days_to_permit,
    julianday(pa.date_certificate_occupancy) - julianday(pa.date_building_permit_issued) AS days_to_final,
    julianday(pa.date_certificate_occupancy) - julianday(pa.date_submitted) AS total_days
FROM permit_applications pa
WHERE pa.date_submitted IS NOT NULL
''')

conn.commit()
print("Created permit_processing_times view")

## 2. Load Existing Projects into Permit Applications

Seed the permit_applications table with our 157 known projects.

In [ ]:
# Load existing projects
projects_df = pd.read_sql('''
    SELECT project_id, address_display, net_units, status, data_source
    FROM projects
    WHERE address_display IS NOT NULL
''', conn)

print(f"Loaded {len(projects_df)} projects")
projects_df.head()

In [ ]:
# Check if we already have permit applications
existing = pd.read_sql('SELECT COUNT(*) as count FROM permit_applications', conn)
print(f"Existing permit applications: {existing['count'].iloc[0]}")

if existing['count'].iloc[0] == 0:
    # Seed with existing projects
    for _, row in projects_df.iterrows():
        cursor.execute('''
            INSERT INTO permit_applications 
            (project_id, street_address, project_name, current_status, data_source)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            row['project_id'],
            row['address_display'],
            row['address_display'],  # Use address as project name initially
            row['status'],
            row['data_source']
        ))
    
    conn.commit()
    print(f"Seeded {len(projects_df)} permit application records")
else:
    print("Permit applications already seeded")

## 3. Building Eye Data Collection

Building Eye provides CSV export of Berkeley permits. This section downloads and parses that data.

In [ ]:
# Check if Building Eye CSV exists (user must download manually)
BUILDING_EYE_CSV = ROOT / 'data' / 'reference' / 'building_eye_permits.csv'

if BUILDING_EYE_CSV.exists():
    be_df = pd.read_csv(BUILDING_EYE_CSV)
    print(f"Loaded {len(be_df)} permits from Building Eye")
    print(f"Columns: {list(be_df.columns)}")
else:
    print("Building Eye CSV not found.")
    print("\nTo collect this data:")
    print("1. Visit https://berkeley.buildingeye.com/")
    print("2. Set filters for housing/residential permits")
    print("3. Click 'Download CSV'")
    print(f"4. Save to: {BUILDING_EYE_CSV}")

## 4. Accela Permit Lookup

For each known project address, search Accela for permit records.

**Note**: This requires careful rate limiting and may need manual verification.

In [ ]:
def search_accela_permits(address):
    """
    Search Berkeley Accela portal for permits at an address.
    Returns list of permit records found.
    
    NOTE: This is a placeholder - actual implementation requires
    navigating the Accela portal which may have rate limits.
    """
    # Accela search URL pattern
    base_url = "https://aca-prod.accela.com/BERKELEY/Cap/CapHome.aspx"
    
    # For now, return empty - manual lookup required
    return {
        'address': address,
        'permits_found': [],
        'search_url': f"{base_url}?module=Building",
        'status': 'manual_lookup_required'
    }

# Test with one address
test_result = search_accela_permits("2120 Allston Way")
print(test_result)

In [ ]:
# Generate lookup list for manual Accela searches
lookup_df = pd.read_sql('''
    SELECT 
        pa.permit_id,
        pa.street_address,
        pa.current_status,
        p.net_units
    FROM permit_applications pa
    JOIN projects p ON pa.project_id = p.project_id
    WHERE pa.date_building_permit_issued IS NULL
    ORDER BY p.net_units DESC
    LIMIT 20
''', conn)

print("Top 20 projects needing permit lookup:")
print("\nVisit: https://aca-prod.accela.com/BERKELEY/")
print("Search each address and record permit dates.\n")

for _, row in lookup_df.iterrows():
    print(f"- {row['street_address']} ({row['net_units']} units) - {row['current_status']}")

## 5. Manual Permit Data Entry

Enter permit timeline data collected from Accela or other sources.

In [ ]:
def update_permit_dates(permit_id, dates_dict):
    """
    Update permit application with timeline dates.
    
    dates_dict can contain:
        - date_submitted
        - date_deemed_complete  
        - date_entitlement_issued
        - date_building_permit_issued
        - date_certificate_occupancy
        - local_tracking_id (permit number)
        - apn
    """
    set_clauses = []
    values = []
    
    for field, value in dates_dict.items():
        if value is not None:
            set_clauses.append(f"{field} = ?")
            values.append(value)
    
    if set_clauses:
        set_clauses.append("updated_at = CURRENT_TIMESTAMP")
        sql = f"UPDATE permit_applications SET {', '.join(set_clauses)} WHERE permit_id = ?"
        values.append(permit_id)
        
        cursor.execute(sql, values)
        conn.commit()
        print(f"Updated permit_id {permit_id}")

# Example usage:
# update_permit_dates(1, {
#     'local_tracking_id': 'BP-2024-12345',
#     'date_submitted': '2024-01-15',
#     'date_deemed_complete': '2024-02-01',
#     'date_building_permit_issued': '2024-06-15'
# })

In [ ]:
def add_affordability_data(project_id, permit_id, affordability_dict):
    """
    Add affordability tracking for a project.
    
    affordability_dict can contain:
        - units_very_low, units_low, units_moderate, units_above_moderate
        - tenure ('rental' or 'ownership')
        - deed_restricted (True/False)
        - affordability_term_years
        - funding_source
    """
    cursor.execute('''
        INSERT INTO affordability_tracking 
        (project_id, permit_id, units_very_low, units_low, units_moderate, 
         units_above_moderate, tenure, deed_restricted, affordability_term_years, funding_source)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        project_id,
        permit_id,
        affordability_dict.get('units_very_low', 0),
        affordability_dict.get('units_low', 0),
        affordability_dict.get('units_moderate', 0),
        affordability_dict.get('units_above_moderate', 0),
        affordability_dict.get('tenure'),
        affordability_dict.get('deed_restricted', False),
        affordability_dict.get('affordability_term_years'),
        affordability_dict.get('funding_source')
    ))
    conn.commit()
    print(f"Added affordability tracking for project {project_id}")

# Example:
# add_affordability_data(1, 1, {
#     'units_very_low': 20,
#     'units_low': 30,
#     'units_above_moderate': 150,
#     'tenure': 'rental',
#     'deed_restricted': True,
#     'affordability_term_years': 55,
#     'funding_source': 'LIHTC'
# })

## 6. Export APR Data

In [ ]:
def export_apr_table_a2(year, output_path=None):
    """
    Export APR Table A2 data for a specific reporting year.
    """
    query = f'''
        SELECT *
        FROM apr_table_a2_view
        WHERE strftime('%Y', "Date Building Permit Issued") = '{year}'
           OR strftime('%Y', "Date Certificate of Occupancy") = '{year}'
           OR strftime('%Y', "Date Entitlement Issued") = '{year}'
    '''
    
    df = pd.read_sql(query, conn)
    
    if output_path:
        df.to_excel(output_path, index=False)
        print(f"Exported {len(df)} records to {output_path}")
    
    return df

# Example:
# apr_2024 = export_apr_table_a2('2024', ROOT / 'outputs' / 'APR_2024_TableA2.xlsx')

In [ ]:
# Summary statistics
summary = pd.read_sql('''
    SELECT 
        COUNT(*) as total_permits,
        SUM(CASE WHEN date_submitted IS NOT NULL THEN 1 ELSE 0 END) as has_submitted_date,
        SUM(CASE WHEN date_building_permit_issued IS NOT NULL THEN 1 ELSE 0 END) as has_permit_date,
        SUM(CASE WHEN date_certificate_occupancy IS NOT NULL THEN 1 ELSE 0 END) as has_co_date,
        SUM(CASE WHEN local_tracking_id IS NOT NULL THEN 1 ELSE 0 END) as has_permit_number
    FROM permit_applications
''', conn)

print("Permit Data Completeness:")
print(f"  Total permits tracked: {summary['total_permits'].iloc[0]}")
print(f"  With submitted date: {summary['has_submitted_date'].iloc[0]}")
print(f"  With permit date: {summary['has_permit_date'].iloc[0]}")
print(f"  With CO date: {summary['has_co_date'].iloc[0]}")
print(f"  With permit number: {summary['has_permit_number'].iloc[0]}")
print("\nNext: Collect dates from Accela portal")

In [ ]:
conn.close()
print("\nDatabase tables created. Ready for data collection.")